In [10]:
"""
CS587 Final Project – Phase 1 (Waterfall)
GenAI-Assisted Project Planning for a Personalized Learning Management System (LMS)

Framework: AutoGen (pyautogen) – GroupChat with GPT-4o-mini
Experiment: Multi-Agent Simulation for Waterfall SDLC Planning
"""

'\nCS587 Final Project – Phase 1 (Waterfall)\nGenAI-Assisted Project Planning for a Personalized Learning Management System (LMS)\n\nFramework: AutoGen (pyautogen) – GroupChat with GPT-4o-mini\nExperiment: Multi-Agent Simulation for Waterfall SDLC Planning\n'

## CS587 Final Project – Phase 1 (Waterfall)
### GenAI-Assisted Project Planning for a Personalized Learning Management System (LMS)

**Framework:** AutoGen (pyautogen) – GroupChat with GPT-4o-mini
**Experiment:** Multi-Agent Simulation for Waterfall SDLC Planning

**AI Agents:**
1. Customer Proxy (UserProxyAgent)
2. Project Manager (GroupChatManager)
3. Requirements Engineer (AssistantAgent)
4. System Engineer (AssistantAgent)
5. Software Engineer (AssistantAgent)
6. Test Engineer (AssistantAgent)
7. Documentation Engineer (AssistantAgent)

**Next Speaker Selection:** Round Robin order

In [11]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "pyautogen<0.4", "openai", "tiktoken", "--quiet"])

import os
import json
import time
from datetime import datetime
import autogen
from autogen import AssistantAgent, UserProxyAgent, GroupChat, GroupChatManager
import tiktoken


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [12]:
env_path = os.path.join(os.getcwd(), ".env")
if os.path.exists(env_path):
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Create a .env file with: OPENAI_API_KEY=sk-...")

MODEL = "gpt-4o-mini"

config_list = [{
    "model": MODEL,
    "api_key": OPENAI_API_KEY,
}]

llm_config = {
    "config_list": config_list,
    "temperature": 0.7,
    "max_tokens": 4096,
}

# Token counter for accurate token measurement
enc = tiktoken.encoding_for_model(MODEL)

print(f"API Key loaded successfully. Model: {MODEL}")
print(f"Framework: AutoGen (pyautogen v{autogen.__version__})")

API Key loaded successfully. Model: gpt-4o-mini
Framework: AutoGen (pyautogen v0.3.2)


## Define AI Agents

Each agent is an AutoGen `AssistantAgent` with a system prompt that defines its role
and responsibilities in the Waterfall SDLC. The `UserProxyAgent` represents the customer.

In [13]:
customer_proxy = UserProxyAgent(
    name="Customer_Proxy",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=0,
    code_execution_config=False,
    default_auto_reply="",
)

requirement_engineer = AssistantAgent(
    name="Requirement_Engineer",
    system_message="""You are a Requirements Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a COMPLETE use case list with unique IDs (UC-001, UC-002, etc.) — provide at least 15-20 use cases
2. For each use case, provide: Use Case ID, Use Case Name, Primary Actor, Preconditions, Postconditions, Main Flow, and Alternate Flows
3. Create detailed FUNCTIONAL requirements (FR-001, FR-002, etc.) — at least 40+ requirements organized by category
4. Create NON-FUNCTIONAL requirements (NFR-001, NFR-002, etc.) — at least 15+ requirements

Then calculate the effort:
- step 1: work = total number of requirements documented
- step 2: productivity rate = 5 requirements completed every day
- step 3: effort = work / productivity

Provide everything in a well-structured markdown format with tables where appropriate.""",
    llm_config=llm_config,
)

system_engineer = AssistantAgent(
    name="System_Engineer",
    system_message="""You are a System Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a HIGH-LEVEL SYSTEM ARCHITECTURE with clear layered design (Client, API/Orchestration, AI/Data Intelligence, Storage, Cross-Cutting)
2. Provide a COMPONENT DIAGRAM (textual description)
3. Provide a SEQUENCE DIAGRAM for a key use case (textual description with numbered steps)
4. Create an ENTITY-RELATIONSHIP DIAGRAM (textual) with all entities, attributes, and relationships
5. Describe the TECHNOLOGY STACK recommendations

Then calculate the effort:
- step 1: work = estimated total number of pages in the design document
- step 2: productivity rate = 5 pages completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format.""",
    llm_config=llm_config,
)

software_engineer = AssistantAgent(
    name="Software_Engineer",
    system_message="""You are a Software Developer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create a detailed IMPLEMENTATION TASK LIST with task IDs (T1, T2, T3...)
2. For each task, provide: description, estimated hours, and dependencies
3. Group tasks by category (Foundation & Infrastructure, Core Backend, Core Frontend, GenAI Integration, Personalization, Admin & Analytics, Quality/Security/Docs)
4. Provide a DEPENDENCY OVERVIEW showing the build order
5. Describe the DEVELOPMENT APPROACH in sequential phases

Then calculate the effort:
- step 1: work = estimated total number of source lines of code (SLOC)
- step 2: productivity rate = 50 SLOC completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format with tables.""",
    llm_config=llm_config,
)

test_engineer = AssistantAgent(
    name="Test_Engineer",
    system_message="""You are a Test Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Define the overall TEST STRATEGY (testing levels: unit, integration, system, AI evaluation)
2. Create FUNCTIONAL TEST CASES (TC-01 through TC-10+) with: Related UCs/FRs, Preconditions, Steps, Expected Results
3. Create NON-FUNCTIONAL TEST CASES (NTC-01 through NTC-06+) covering: Performance, AI Latency, Availability, Security/RBAC, Usability, Data Privacy
4. Provide a TESTING SCHEDULE broken into weeks

Then calculate the effort:
- step 1: work = estimated total number of test cases
- step 2: productivity rate = 2 test cases completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format with tables.""",
    llm_config=llm_config,
)

documentation_engineer = AssistantAgent(
    name="Documentation_Engineer",
    system_message="""You are a Documentation Engineer AI Agent for a software project using the Waterfall development model.

Your responsibilities:
1. Create an END-USER DOCUMENTATION OUTLINE (for students, instructors, admins)
2. Create a DEVELOPER DOCUMENTATION OUTLINE (for engineers)
3. Provide a DOCUMENTATION SCHEDULE showing pages per day and weekly breakdown
4. Estimate total pages for user docs and developer docs

Then calculate the effort:
- step 1: work = estimated total number of documentation pages
- step 2: productivity rate = 3 pages completed every day
- step 3: effort = work / productivity

Provide everything in well-structured markdown format.""",
    llm_config=llm_config,
)

print("All agents created successfully.")
print("  - Customer_Proxy (UserProxyAgent)")
print("  - Requirement_Engineer (AssistantAgent)")
print("  - System_Engineer (AssistantAgent)")
print("  - Software_Engineer (AssistantAgent)")
print("  - Test_Engineer (AssistantAgent)")
print("  - Documentation_Engineer (AssistantAgent)")

All agents created successfully.
  - Customer_Proxy (UserProxyAgent)
  - Requirement_Engineer (AssistantAgent)
  - System_Engineer (AssistantAgent)
  - Software_Engineer (AssistantAgent)
  - Test_Engineer (AssistantAgent)
  - Documentation_Engineer (AssistantAgent)


## Customer Problem Statement & GroupChat Setup

The customer sends the initial requirements. AutoGen's `GroupChat` with **round-robin**
speaker selection ensures each engineer agent responds in the correct Waterfall order.
The `GroupChatManager` orchestrates the conversation (acting as the Project Manager).

In [14]:
customer_message = (
    "I want to build a GenAI-Assisted Personalized Learning Management System (LMS) "
    "with the following core objectives and capabilities:\n\n"
    "1. Personalized Learning Paths for Students: Use GenAI to analyze a student's "
    "goals, current knowledge, performance, and preferences. Automatically generate "
    "and continuously adapt personalized learning paths (modules, lessons, assessments, "
    "and projects). Recommend content, practice questions, and projects tailored to "
    "each student.\n\n"
    "2. AI Tutor for Students: Provide an AI tutor chatbot inside the LMS that can "
    "answer course-related questions in natural language, explain concepts at different "
    "difficulty levels (beginner to advanced), generate examples, quizzes, and hints, "
    "and suggest next steps or remedial content when students are stuck.\n\n"
    "3. AI Assistance for Instructors: Help instructors create course materials "
    "including lecture outlines, slide drafts, reading summaries, quiz questions, "
    "assignments, rubrics, and sample answers. Allow instructors to input a topic or "
    "syllabus and have the system propose structured modules, learning objectives, "
    "and suggested assessments.\n\n"
    "4. Progress Tracking Dashboards: Dashboards for students showing learning path "
    "progress, completed modules, scores, and identified weak areas with AI-generated "
    "insights and recommendations. Dashboards for instructors/admins with class-level "
    "analytics, engagement, completion rates, average performance by topic, and "
    "individual student insights including risk of falling behind.\n\n"
    "5. Automated Project Planning Support Using AI Agents: Include AI agents that "
    "help students and/or instructors plan projects by breaking down a project into "
    "phases, tasks, and milestones; suggesting timelines, deliverables, and checklists; "
    "and adjusting plans based on progress updates.\n\n"
    "6. General Expectations: The system should be designed as a modern web application, "
    "architected to integrate with one or more LLM/GenAI providers (e.g., OpenAI or "
    "local models), with security, privacy, and role-based access for students, "
    "instructors, and admins."
)

print("Customer problem statement loaded.")
print(f"Message length: {len(customer_message)} characters")

Customer problem statement loaded.
Message length: 2045 characters


In [15]:
agents = [
    customer_proxy,
    requirement_engineer,
    system_engineer,
    software_engineer,
    test_engineer,
    documentation_engineer,
]

groupchat = GroupChat(
    agents=agents,
    messages=[],
    max_round=6,
    speaker_selection_method="round_robin",
    allow_repeat_speaker=False,
)

manager = GroupChatManager(
    groupchat=groupchat,
    llm_config=llm_config,
)

print("GroupChat configured:")
print(f"  Speaker selection: round_robin")
print(f"  Max rounds: {groupchat.max_round}")
print(f"  Agents: {[a.name for a in agents]}")

GroupChat configured:
  Speaker selection: round_robin
  Max rounds: 6
  Agents: ['Customer_Proxy', 'Requirement_Engineer', 'System_Engineer', 'Software_Engineer', 'Test_Engineer', 'Documentation_Engineer']


## Run Experiment

Execute the full Waterfall project planning simulation. The `Customer_Proxy` sends
the initial requirements, and AutoGen's `GroupChatManager` orchestrates the round-robin
conversation through all engineering agents.

In [16]:
# Clear any previous messages to force a fresh run
groupchat.messages.clear()
print("Starting fresh GroupChat experiment...")

start_time = time.time()

# Kick off the AutoGen GroupChat
chat_result = customer_proxy.initiate_chat(
    manager,
    message=customer_message,
)

total_elapsed = time.time() - start_time

print(f"\n{'=' * 60}")
print(f"  GroupChat completed in {total_elapsed:.1f}s")
print(f"{'=' * 60}")

Starting fresh GroupChat experiment...
Customer_Proxy (to chat_manager):

I want to build a GenAI-Assisted Personalized Learning Management System (LMS) with the following core objectives and capabilities:

1. Personalized Learning Paths for Students: Use GenAI to analyze a student's goals, current knowledge, performance, and preferences. Automatically generate and continuously adapt personalized learning paths (modules, lessons, assessments, and projects). Recommend content, practice questions, and projects tailored to each student.

2. AI Tutor for Students: Provide an AI tutor chatbot inside the LMS that can answer course-related questions in natural language, explain concepts at different difficulty levels (beginner to advanced), generate examples, quizzes, and hints, and suggest next steps or remedial content when students are stuck.

3. AI Assistance for Instructors: Help instructors create course materials including lecture outlines, slide drafts, reading summaries, quiz questio

## Save Results

In [17]:
output_dir = "Phase1_Experiment_Results"
os.makedirs(output_dir, exist_ok=True)

agent_names = [
    "Requirement_Engineer",
    "System_Engineer",
    "Software_Engineer",
    "Test_Engineer",
    "Documentation_Engineer",
]

# System prompts for token counting
agent_system_prompts = {
    a.name: a.system_message for a in agents if hasattr(a, 'system_message') and a.system_message
}

all_results = []
for msg in groupchat.messages:
    name = msg.get("name", "")
    content = msg.get("content", "")
    if name in agent_names and content:
        # Count tokens using tiktoken (accurate for OpenAI models)
        output_tokens = len(enc.encode(content))
        sys_prompt = agent_system_prompts.get(name, "")
        input_tokens = len(enc.encode(sys_prompt)) + len(enc.encode(customer_message))
        total_tokens_agent = input_tokens + output_tokens

        all_results.append({
            "agent": name,
            "response": content,
            "tokens": total_tokens_agent,
            "model": MODEL,
        })

# Calculate total tokens across all agents
total_tokens = sum(r['tokens'] for r in all_results)

# Save individual agent outputs
for result in all_results:
    filepath = os.path.join(output_dir, f"{result['agent']}_output.md")
    with open(filepath, "w") as f:
        f.write(f"# {result['agent'].replace('_', ' ')} Output\n\n")
        f.write(f"**Model:** {result['model']}\n\n")
        f.write(f"**Tokens Used:** {result['tokens']}\n\n")
        f.write("---\n\n")
        f.write(result['response'])
    print(f"Saved: {filepath}")

# Combined report
combined_path = os.path.join(output_dir, "Phase1_Waterfall_Complete_Report.md")
section_names = {
    "Requirement_Engineer": "SECTION 2 — REQUIREMENTS ENGINEER AI AGENT",
    "System_Engineer": "SECTION 3 — SYSTEM ENGINEER AI AGENT",
    "Software_Engineer": "SECTION 4 — SOFTWARE DEVELOPER AI AGENT",
    "Test_Engineer": "SECTION 5 — TEST ENGINEER AI AGENT",
    "Documentation_Engineer": "SECTION 6 — DOCUMENTATION ENGINEER AI AGENT",
}
with open(combined_path, "w") as f:
    f.write("# Phase 1 – Waterfall Project Plan\n")
    f.write("# GenAI-Assisted Personalized Learning Management System (LMS)\n\n")
    f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    f.write(f"**LLM Model:** {MODEL}\n\n")
    f.write("**Framework:** AutoGen (pyautogen) – GroupChat Multi-Agent Simulation\n\n")
    f.write("---\n\n")
    f.write("## SECTION 1 — CUSTOMER PROXY & PROJECT MANAGER\n\n")
    f.write("### 1.1 Customer Problem Statement\n\n")
    f.write(customer_message.strip())
    f.write("\n\n---\n\n")
    for result in all_results:
        agent = result['agent']
        if agent in section_names:
            f.write(f"## {section_names[agent]}\n\n")
            f.write(f"**Model:** {result['model']} | **Tokens:** {result['tokens']:,}\n\n")
            f.write(result['response'])
            f.write("\n\n---\n\n")
print(f"Saved combined report: {combined_path}")

# Metadata
metadata = {
    "experiment": {
        "title": "Phase 1 – Waterfall Project Planning",
        "project": "GenAI-Assisted Personalized LMS",
        "model": MODEL,
        "framework": f"AutoGen (pyautogen v{autogen.__version__}) – GroupChat",
        "speaker_selection": "round_robin",
        "date": datetime.now().isoformat(),
        "total_tokens": total_tokens,
        "total_time_seconds": round(total_elapsed, 2),
    },
    "agents": [{"name": r['agent'], "tokens": r['tokens'], "model": r['model']} for r in all_results],
    "groupchat": {
        "total_messages": len(groupchat.messages),
        "max_round": groupchat.max_round,
    },
}
metadata_path = os.path.join(output_dir, "experiment_metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)
print(f"Saved metadata: {metadata_path}")

Saved: Phase1_Experiment_Results/Requirement_Engineer_output.md
Saved: Phase1_Experiment_Results/System_Engineer_output.md
Saved: Phase1_Experiment_Results/Software_Engineer_output.md
Saved: Phase1_Experiment_Results/Test_Engineer_output.md
Saved: Phase1_Experiment_Results/Documentation_Engineer_output.md
Saved combined report: Phase1_Experiment_Results/Phase1_Waterfall_Complete_Report.md
Saved metadata: Phase1_Experiment_Results/experiment_metadata.json


## Experiment Summary

In [18]:
print(f"{'=' * 60}")
print(f"  EXPERIMENT COMPLETE")
print(f"{'=' * 60}")
print()
print(f"  Model:         {MODEL}")
print(f"  Framework:     AutoGen (pyautogen v{autogen.__version__})")
print(f"  GroupChat:     round_robin")
print(f"  Total Agents:  {len(all_results)}")
print(f"  Total Rounds:  {len(groupchat.messages)}")
print(f"  Total Tokens:  {total_tokens:,}")
print(f"  Total Time:    {total_elapsed:.1f}s")
print()
print(f"  Agent Breakdown:")
for r in all_results:
    print(f"    {r['agent']:30s} | Tokens: {r['tokens']:6,} | Response: {len(r['response']):6,} chars")
print()
print(f"  Output Directory: {output_dir}/")
print(f"{'=' * 60}")

  EXPERIMENT COMPLETE

  Model:         gpt-4o-mini
  Framework:     AutoGen (pyautogen v0.3.2)
  GroupChat:     round_robin
  Total Agents:  5
  Total Rounds:  6
  Total Tokens:  9,442
  Total Time:    151.1s

  Agent Breakdown:
    Requirement_Engineer           | Tokens:  2,274 | Response: 10,041 chars
    System_Engineer                | Tokens:  2,003 | Response:  6,537 chars
    Software_Engineer              | Tokens:  2,027 | Response:  6,949 chars
    Test_Engineer                  | Tokens:  1,857 | Response:  6,707 chars
    Documentation_Engineer         | Tokens:  1,281 | Response:  3,300 chars

  Output Directory: Phase1_Experiment_Results/
